In [1]:
!pip install -q vllm openai requests

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 whic

In [3]:
%%writefile evaluation_data.json
[
  {"prompt": "What is the capital of France?", "expected": "Paris"},
  {"prompt": "What is 12 * 8?", "expected": "96"},
  {"prompt": "Who wrote Romeo and Juliet?", "expected": "Shakespeare"},
  {"prompt": "What is the chemical symbol for water?", "expected": "H2O"},
  {"prompt": "What is the largest planet in our solar system?", "expected": "Jupiter"},
  {"prompt": "What is the capital of Japan?", "expected": "Tokyo"},
  {"prompt": "What is 15 + 27?", "expected": "42"},
  {"prompt": "Who painted the Mona Lisa?", "expected": "Leonardo"},
  {"prompt": "What is the square root of 81?", "expected": "9"},
  {"prompt": "What gas do plants absorb from the air?", "expected": "carbon dioxide"},
  {"prompt": "What is the capital of Italy?", "expected": "Rome"},
  {"prompt": "How many days are in a week?", "expected": "7"},
  {"prompt": "What is the boiling point of water in Celsius?", "expected": "100"},
  {"prompt": "Who discovered gravity after seeing an apple fall?", "expected": "Newton"},
  {"prompt": "What is the capital of Germany?", "expected": "Berlin"},
  {"prompt": "What is 9 * 7?", "expected": "63"},
  {"prompt": "What is the smallest prime number?", "expected": "2"},
  {"prompt": "Which planet is known as the Red Planet?", "expected": "Mars"},
  {"prompt": "What language is primarily spoken in Brazil?", "expected": "Portuguese"},
  {"prompt": "What is the chemical symbol for gold?", "expected": "Au"}
]

Overwriting evaluation_data.json


In [4]:
import subprocess

import time, json, requests, glob
from openai import OpenAI

DATA = "evaluation_data.json"
data = json.load(open(DATA, encoding="utf-8"))

# FP16 vs 8-bit vs 4-bit (official Qwen repos)
MODELS = [
    ("FP16",      "Qwen/Qwen2.5-1.5B-Instruct",           []),
    ("8-bit",     "Qwen/Qwen2.5-1.5B-Instruct-GPTQ-Int8", ["--quantization", "gptq"]),
    ("4-bit AWQ", "Qwen/Qwen2.5-1.5B-Instruct-AWQ",       ["--quantization", "awq"]),
]

def stop_all():
    subprocess.run(["pkill", "-f", "vllm serve"])
    time.sleep(15)

def start(model, extra):
    proc = subprocess.Popen(
        ["vllm", "serve", model, "--port", "8000", "--dtype", "half",
         "--max-model-len", "4096", "--gpu-memory-utilization", "0.8"] + extra,
        stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(600):
        try:
            if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
                return proc
        except requests.RequestException:
            pass
        time.sleep(2)
    raise RuntimeError(f"{model} did not start, check vllm.log")

def gpu_mem():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.split("\n")[0]
    return int(out)

def run_eval(model):
    client = OpenAI(base_url="http://localhost:8000/v1", api_key="none")
    res = []
    for item in data:
        t = time.time()
        r = client.chat.completions.create(
            model=model, temperature=0,
            messages=[{"role": "system", "content": "You are a helpful assistant. Answer briefly."},
                      {"role": "user", "content": item["prompt"]}])
        ans = r.choices[0].message.content
        res.append({"correct": item["expected"].lower() in ans.lower(),
                    "latency": time.time() - t, "tokens": r.usage.completion_tokens})
    return res

summary = []
for name, model, extra in MODELS:
    print(f"Running {name}...")
    stop_all()
    proc = start(model, extra)
    try:
        run_eval(model)               # warm-up
        mem = gpu_mem()
        res = run_eval(model)
    finally:
        proc.terminate()
    acc = sum(r["correct"] for r in res) / len(res)
    lat = sum(r["latency"] for r in res) / len(res)
    tps = sum(r["tokens"] for r in res) / sum(r["latency"] for r in res)
    summary.append({"version": name, "accuracy": acc, "latency_s": round(lat, 3),
                    "tokens_per_s": round(tps, 1), "gpu_mem_MB": mem})
    print(summary[-1])

json.dump(summary, open("quant_results.json", "w"), indent=2)

Running FP16...
{'version': 'FP16', 'accuracy': 0.9, 'latency_s': 0.149, 'tokens_per_s': 57.8, 'gpu_mem_MB': 11615}
Running 8-bit...
{'version': '8-bit', 'accuracy': 0.9, 'latency_s': 0.1, 'tokens_per_s': 86.3, 'gpu_mem_MB': 11599}
Running 4-bit AWQ...
{'version': '4-bit AWQ', 'accuracy': 0.95, 'latency_s': 0.105, 'tokens_per_s': 112.2, 'gpu_mem_MB': 11629}


In [5]:
!pip install -q colab-vscode-bridge
from colab_vscode_bridge import ColabFileDownloader

ColabFileDownloader("quant_results.json")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.18.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
File: quant_results.json (Size: 0.00 Mb)


ColabFileDownloader(children=(Button(description='Download File', style=ButtonStyle()), IntProgress(value=0, b…

<IPython.core.display.Javascript object>

In [6]:
import subprocess, time, requests

subprocess.run(["pkill", "-f", "vllm serve"]); time.sleep(10)
MODEL = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"
proc = subprocess.Popen(
    ["vllm", "serve", MODEL, "--port", "8000", "--dtype", "half",
     "--max-model-len", "4096", "--gpu-memory-utilization", "0.8", "--quantization", "awq"],
    stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)

for _ in range(600):
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            print("ready"); break
    except requests.RequestException:
        pass
    time.sleep(2)

ready


In [8]:
import json
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="none")
data = json.load(open("evaluation_data.json"))
prompts = [d["prompt"] + " Explain in 2 sentences." for d in data] * 2  # 40 requests

def one(p):
    t = time.time()
    r = client.chat.completions.create(model=MODEL, temperature=0, max_tokens=100,
                                       messages=[{"role": "user", "content": p}])
    return time.time() - t, r.usage.completion_tokens

def run(concurrency):
    start = time.time()
    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        res = list(ex.map(one, prompts))
    total = time.time() - start
    return {"concurrency": concurrency,
            "total_time_s": round(total, 2),
            "avg_latency_s": round(sum(r[0] for r in res) / len(res), 2),
            "tokens_per_s": round(sum(r[1] for r in res) / total, 1)}

In [9]:
run(1)  # warm-up
results = []
for c in [1, 4, 8, 16, 32]:
    results.append(run(c))
    print(results[-1])

{'concurrency': 1, 'total_time_s': 9.66, 'avg_latency_s': 0.24, 'tokens_per_s': 128.4}
{'concurrency': 4, 'total_time_s': 3.05, 'avg_latency_s': 0.3, 'tokens_per_s': 406.3}
{'concurrency': 8, 'total_time_s': 1.64, 'avg_latency_s': 0.32, 'tokens_per_s': 758.4}
{'concurrency': 16, 'total_time_s': 1.17, 'avg_latency_s': 0.41, 'tokens_per_s': 1078.6}
{'concurrency': 32, 'total_time_s': 0.99, 'avg_latency_s': 0.6, 'tokens_per_s': 1267.8}


In [10]:
json.dump(results, open("batching_results.json", "w"), indent=2)
proc.terminate()

In [11]:
# !pip install -q colab-vscode-bridge
# from colab_vscode_bridge import ColabFileDownloader

ColabFileDownloader("batching_results.json")

File: batching_results.json (Size: 0.00 Mb)


ColabFileDownloader(children=(Button(description='Download File', style=ButtonStyle()), IntProgress(value=0, b…

<IPython.core.display.Javascript object>

Cell 1 (server starter)

In [12]:
# import subprocess, time, requests, json
# from openai import OpenAI

MODEL = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"

def start(caching):
    subprocess.run(["pkill", "-f", "vllm serve"]); time.sleep(10)
    flag = "--enable-prefix-caching" if caching else "--no-enable-prefix-caching"
    proc = subprocess.Popen(
        ["vllm", "serve", MODEL, "--port", "8000", "--dtype", "half", "--max-model-len", "4096",
         "--gpu-memory-utilization", "0.8", "--quantization", "awq", flag],
        stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(600):
        try:
            if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
                return proc
        except requests.RequestException:
            pass
        time.sleep(2)
    raise RuntimeError("server did not start, check vllm.log")

Cell 2 (test function)

In [13]:
client = OpenAI(base_url="http://localhost:8000/v1", api_key="none")
context = " ".join(f"Note {i}: the weather report for day {i} was cloudy with light wind." for i in range(120))
questions = [f"What was the weather on day {i}? Answer briefly." for i in range(0, 100, 10)]

def ttft(q):
    t = time.time(); first = None
    stream = client.chat.completions.create(
        model=MODEL, temperature=0, max_tokens=30, stream=True,
        messages=[{"role": "system", "content": "Context:\n" + context},
                  {"role": "user", "content": q}])
    for chunk in stream:
        if first is None and chunk.choices and chunk.choices[0].delta.content:
            first = time.time() - t
    return first

def run(caching):
    proc = start(caching)
    try:
        times = [ttft(q) for q in questions]
    finally:
        proc.terminate()
    return {"prefix_caching": caching,
            "first_request_ttft_s": round(times[0], 3),
            "avg_ttft_after_first_s": round(sum(times[1:]) / len(times[1:]), 3)}

Cell 3 (run both and save)

In [14]:
results = [run(False), run(True)]
for r in results: print(r)
json.dump(results, open("caching_results.json", "w"), indent=2)

{'prefix_caching': False, 'first_request_ttft_s': 2.324, 'avg_ttft_after_first_s': 1.135}
{'prefix_caching': True, 'first_request_ttft_s': 1.289, 'avg_ttft_after_first_s': 0.06}


Cell 4 (download)

In [15]:
# from colab_vscode_bridge import ColabFileDownloader
ColabFileDownloader("caching_results.json")

File: caching_results.json (Size: 0.00 Mb)


ColabFileDownloader(children=(Button(description='Download File', style=ButtonStyle()), IntProgress(value=0, b…

<IPython.core.display.Javascript object>

Cell 1 (start the 4-bit server)

In [16]:
# import subprocess, time, requests, json
# from openai import OpenAI

MODEL = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"
subprocess.run(["pkill", "-f", "vllm serve"]); time.sleep(10)
proc = subprocess.Popen(
    ["vllm", "serve", MODEL, "--port", "8000", "--dtype", "half", "--max-model-len", "4096",
     "--gpu-memory-utilization", "0.8", "--quantization", "awq"],
    stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)

for _ in range(600):
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            print("ready"); break
    except requests.RequestException:
        pass
    time.sleep(2)

client = OpenAI(base_url="http://localhost:8000/v1", api_key="none")

ready


Cell 2 (streaming and non-streaming test functions)

In [17]:
prompts = ["Explain what a KV cache is.",
           "Write a short story about a robot.",
           "Explain quantization in simple words.",
           "Describe how batching speeds up inference."]

def streaming(p):
    t0 = time.time(); first = None; tokens = 0
    stream = client.chat.completions.create(
        model=MODEL, temperature=0, max_tokens=200, stream=True,
        stream_options={"include_usage": True},
        messages=[{"role": "user", "content": p}])
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content and first is None:
            first = time.time()
        if chunk.usage:
            tokens = chunk.usage.completion_tokens
    end = time.time()
    return {"first_visible_s": first - t0, "total_s": end - t0,
            "tokens": tokens, "tokens_per_s": tokens / (end - first)}

def non_streaming(p):
    t0 = time.time()
    r = client.chat.completions.create(
        model=MODEL, temperature=0, max_tokens=200,
        messages=[{"role": "user", "content": p}])
    total = time.time() - t0
    return {"first_visible_s": total, "total_s": total,
            "tokens": r.usage.completion_tokens,
            "tokens_per_s": r.usage.completion_tokens / total}

Cell 3 (run both and save)

In [18]:
streaming(prompts[0]); non_streaming(prompts[0])  # warm-up

def avg(f):
    rs = [f(p) for p in prompts]
    return {k: round(sum(r[k] for r in rs) / len(rs), 3) for k in rs[0]}

results = {"streaming": avg(streaming), "non_streaming": avg(non_streaming)}
for k, v in results.items():
    print(k, v)
json.dump(results, open("streaming_results.json", "w"), indent=2)

streaming {'first_visible_s': 0.03, 'total_s': 1.454, 'tokens': 199.25, 'tokens_per_s': 139.928}
non_streaming {'first_visible_s': 1.464, 'total_s': 1.464, 'tokens': 199.25, 'tokens_per_s': 136.081}


Cell 4 (stop the server and download)

In [19]:
proc.terminate()
# from colab_vscode_bridge import ColabFileDownloader
ColabFileDownloader("streaming_results.json")

File: streaming_results.json (Size: 0.00 Mb)


ColabFileDownloader(children=(Button(description='Download File', style=ButtonStyle()), IntProgress(value=0, b…

<IPython.core.display.Javascript object>

In [ ]:
import subprocess, time, json, requests, os
from openai import OpenAI

# 1. Load evaluation data
if os.path.exists("evaluation_data.json"):
    data = json.load(open("evaluation_data.json", encoding="utf-8"))
else:
    data = [
        {"prompt": "What is the capital of France?", "expected": "Paris"},
        {"prompt": "What is 12 * 8?", "expected": "96"},
        {"prompt": "Who wrote Romeo and Juliet?", "expected": "Shakespeare"},
        {"prompt": "What is the capital of Japan?", "expected": "Tokyo"},
        {"prompt": "What is the square root of 81?", "expected": "9"}
    ]

# 2. Helpers to start/stop vLLM
def stop_all():
    subprocess.run(["pkill", "-f", "vllm serve"])
    time.sleep(10)

def start_server(extra_args=[]):
    cmd = [
        "vllm", "serve", "Qwen/Qwen2.5-1.5B-Instruct",
        "--port", "8000",
        "--dtype", "half",
        "--max-model-len", "2048",
        "--gpu-memory-utilization", "0.8"
    ] + extra_args
    proc = subprocess.Popen(cmd, stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(300):
        try:
            if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
                return proc
        except requests.RequestException:
            pass
        time.sleep(2)
    raise RuntimeError("vLLM failed to start. Check vllm.log for details.")

def run_eval():
    client = OpenAI(base_url="http://localhost:8000/v1", api_key="none")
    res = []
    for item in data:
        t = time.time()
        r = client.chat.completions.create(
            model="Qwen/Qwen2.5-1.5B-Instruct",
            temperature=0,
            messages=[{"role": "user", "content": item["prompt"]}]
        )
        res.append({
            "latency": time.time() - t,
            "tokens": r.usage.completion_tokens
        })
    avg_lat = sum(r["latency"] for r in res) / len(res)
    tps = sum(r["tokens"] for r in res) / sum(r["latency"] for r in res)
    return round(tps, 1), round(avg_lat, 3)

# 3. Benchmark Baseline vs Speculative Decoding
CONFIGS = [
    ("Baseline (No Speculation)", []),
    ("Speculative Decoding (0.5B Draft)", [
        "--speculative-config", '{"method": "draft_model","model": "Qwen/Qwen2.5-0.5B-Instruct", "num_speculative_tokens": 5}'
    ])
]

results = []
for name, extra in CONFIGS:
    print(f"Running {name}...")
    stop_all()
    proc = start_server(extra)
    try:
        run_eval()  # warm-up
        tps, lat = run_eval()
        results.append({"name": name, "tokens_per_s": tps, "latency_s": lat})
        print(f"-> {name}: {tps} tokens/s, avg latency: {lat}s")
    finally:
        proc.terminate()
        stop_all()

# 4. Save results to JSON
json.dump(results, open("speculative_results.json", "w"), indent=2)
print("\nDone! Saved to speculative_results.json")

In [ ]:
import subprocess, time, json, requests, os
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI

# 1. Prompts for testing (replicated to 40 requests to measure tail distribution)
base_prompts = [
    "What is the capital of France?",
    "Explain what a KV cache is in two sentences.",
    "Write a quick tip for learning Python.",
    "What is 25 * 14?",
    "Describe the role of GPU in deep learning."
]
requests_list = (base_prompts * 8)[:40]  # 40 total requests

# 2. Helpers to start/stop vLLM
def stop_all():
    subprocess.run(["pkill", "-f", "vllm serve"])
    time.sleep(10)

def start_server():
    cmd = [
        "vllm", "serve", "Qwen/Qwen2.5-1.5B-Instruct",
        "--port", "8000",
        "--dtype", "half",
        "--max-model-len", "2048",
        "--gpu-memory-utilization", "0.8"
    ]
    proc = subprocess.Popen(cmd, stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(300):
        try:
            if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
                return proc
        except requests.RequestException:
            pass
        time.sleep(2)
    raise RuntimeError("vLLM failed to start. Check vllm.log")

# 3. Request function measuring TTFT (Time to First Token) and Total Latency
def measure_streaming_request(prompt):
    client = OpenAI(base_url="http://localhost:8000/v1", api_key="none")
    t0 = time.time()
    first_token_time = None
    stream = client.chat.completions.create(
        model="Qwen/Qwen2.5-1.5B-Instruct",
        temperature=0,
        max_tokens=80,
        stream=True,
        messages=[{"role": "user", "content": prompt}]
    )
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content and first_token_time is None:
            first_token_time = time.time()
    tend = time.time()
    ttft = (first_token_time - t0) if first_token_time else (tend - t0)
    total_latency = tend - t0
    return {"ttft": ttft, "total_latency": total_latency}

# 4. Run Benchmark under Concurrency (8 parallel clients)
print("Starting vLLM server for latency distribution benchmark...")
stop_all()
proc = start_server()

try:
    print("Warming up server...")
    measure_streaming_request("Hello")
    
    print(f"Sending {len(requests_list)} concurrent requests (Concurrency = 8)...")
    with ThreadPoolExecutor(max_workers=8) as executor:
        records = list(executor.map(measure_streaming_request, requests_list))
finally:
    proc.terminate()
    stop_all()

# 5. Calculate Percentiles (p50, p90, p95, p99)
ttfts = [r["ttft"] for r in records]
totals = [r["total_latency"] for r in records]

percentiles_summary = {
    "total_requests": len(records),
    "concurrency": 8,
    "ttft_seconds": {
        "mean": round(float(np.mean(ttfts)), 3),
        "p50": round(float(np.percentile(ttfts, 50)), 3),
        "p90": round(float(np.percentile(ttfts, 90)), 3),
        "p95": round(float(np.percentile(ttfts, 95)), 3),
        "p99": round(float(np.percentile(ttfts, 99)), 3),
    },
    "total_latency_seconds": {
        "mean": round(float(np.mean(totals)), 3),
        "p50": round(float(np.percentile(totals, 50)), 3),
        "p90": round(float(np.percentile(totals, 90)), 3),
        "p95": round(float(np.percentile(totals, 95)), 3),
        "p99": round(float(np.percentile(totals, 99)), 3),
    }
}

print("\n--- Latency & TTFT Percentiles ---")
print(json.dumps(percentiles_summary, indent=2))

# 6. Save results to JSON
json.dump(percentiles_summary, open("latency_percentiles.json", "w"), indent=2)
print("\nSaved to latency_percentiles.json")